In [13]:
import pandas as pd
import numpy as np

df = pd.read_excel(r"D:\Projects\Data\Sleep_Work_Productivity_Surve.xlsx")

# clean age column

df['Enter your age ?']= df['Enter your age ?'].fillna(df['Enter your age ?'].median())

# rename the columns

df.rename(columns={
    'How would you rate your overall work productivity?': 'work_productivity',
    'How much do you think your sleep affects your productivity at work?': 'sleep_productivity_effect',
    'What factors do you believe impact your sleep quality? (Select all that apply)': 'sleep_factors',
    'How many breaks do you take during your workday when you are tired ?': 'break_nums',
    'How would you describe your energy levels throughout the workday?': 'energy_levels_per_workday',
    'What is your primary work environment?': 'primary_work_environment',
    'what is you gender ?': 'gender',
    'On a typical work night, how many hours of sleep do you get?': 'sleep_hours',
    
    'I wake up feeling refreshed and ready for work.':
        'wake_up_refreshed',

    'I have difficulty falling asleep the night before an important workday.':
        'difficulty_falling_asleep',

    'Enter your age ?':
        'age',

    'I wake up multiple times during the night and struggle to go back to sleep.':
        'wake_up_multiple_times',

    'On weekends or days off, my sleep schedule is very different from my workday schedule (e.g., sleeping in more than 2 hours later).':
        'weekend_sleep_schedule_difference',

    'I use electronic devices (phone, laptop, TV) in bed within 30 minutes of trying to sleep.':
        'device_use_before_sleep',

    'I find it hard to concentrate on work tasks for more than 15–20 minutes at a time.':
        'difficulty_concentrating',

    'I make careless mistakes (e.g., typos, miscalculations, forgetting steps) that I would not have made if better rested.':
        'careless_mistakes',

    'I often have to re-read emails, documents, or instructions because I lost focus.':
        'reread_due_to_focus_loss',

    'Learning a new software, process, or skill at work feels unusually difficult for me.':
        'difficulty_learning_new_skills',

    'I struggle to generate new ideas or creative solutions to problems.':
        'difficulty_generating_ideas',

    'I forget important tasks, deadlines, or details from meetings.':
        'forget_tasks_deadlines',

    'I feel irritable or short-tempered with colleagues or clients.':
        'irritable_with_colleagues',

    'Small setbacks or minor criticism feel overwhelming or upsetting.':
        'small_setbacks_overwhelming',

    'I avoid collaboration or group work because I feel mentally exhausted.':
        'avoid_collaboration',

    'I have difficulty controlling my emotions during stressful work situations (e.g., tight deadlines, difficult customers).':
        'difficulty_controlling_emotions',

    'I feel physically tired, sluggish, or heavy-eyed during work, even without heavy physical exertion.':
        'physically_tired_at_work',

    'In my job, I have had a close call, minor accident, or injury that I attribute to being too tired. If your job is sedentary, answer based on accidents like tripping, bumping into things, or spilling hot liquids.':
        'accident_due_to_tiredness',

    'Compared to when I am well-rested, I estimate my current work output is...':
        'current_work_output',

    'In the past month, how many work days did you feel you were “present but not productive” (getting less than half of your normal output) due to poor sleep?':
        'unproductive_workdays_due_to_sleep'

}, inplace=True)


df = df.replace(r'\n', ', ', regex=True)
df["sleep_factors"] = df["sleep_factors"].str.split(", ")

# clean unproductive_workdays_due_to_sleep outlires

df.loc[df["unproductive_workdays_due_to_sleep"] > 23,
       "unproductive_workdays_due_to_sleep"] = np.nan

median = df['unproductive_workdays_due_to_sleep'].median().round(0)
df['unproductive_workdays_due_to_sleep'] = df['unproductive_workdays_due_to_sleep'].fillna(median)


In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 127 entries, 0 to 126
Data columns (total 29 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Submission Date                     127 non-null    str    
 1   work_productivity                   127 non-null    str    
 2   sleep_productivity_effect           127 non-null    str    
 3   sleep_factors                       127 non-null    object 
 4   break_nums                          122 non-null    str    
 5   energy_levels_per_workday           127 non-null    str    
 6   primary_work_environment            125 non-null    str    
 7   gender                              127 non-null    str    
 8   sleep_hours                         127 non-null    str    
 9   wake_up_refreshed                   127 non-null    int64  
 10  difficulty_falling_asleep           127 non-null    int64  
 11  age                                 127 non-null    floa

In [ ]:
#Reverse the only positive-direction question
df['wake_up_refreshed_reverse'] = 6 - df['wake_up_refreshed']


# 1. Sleep Quality and Hygiene Index (SQLI)
# 5 questions × maximum score of 5 = 25
# Divide by 25 and multiply by 100 → maximum possible score = 100
df['sleep_quality_and_hygiene_index_(SQLI)'] = (
    (
        df['wake_up_refreshed_reverse']
        + df['difficulty_falling_asleep']
        + df['wake_up_multiple_times']
        + df['weekend_sleep_schedule_difference']
        + df['device_use_before_sleep']
    ) / 25
) * 100


# 2. Cognitive at-task Performance Index (CTPI)
# 6 questions × maximum score of 5 = 30
# Divide by 30 and multiply by 100 → maximum possible score = 100
df['Cognitive_at_task_perforamnce_index_(CTPI)'] = (
    (df['difficulty_concentrating']
     + df['careless_mistakes']
     + df['reread_due_to_focus_loss']
     + df['difficulty_learning_new_skills']
     + df['difficulty_generating_ideas']
     + df['forget_tasks_deadlines']) / 30
) * 100

df['Cognitive_at_task_perforamnce_index_(CTPI)'] = (
    df['Cognitive_at_task_perforamnce_index_(CTPI)'].round(2)
)


# 3. Emotional and Social Impact Index (ESII)
# 4 questions × maximum score of 5 = 20
# Divide by 20 and multiply by 100 → maximum possible score = 100
df['Emotional_and_social_impact_index(ESII)'] = (
    (df['irritable_with_colleagues']
     + df['small_setbacks_overwhelming']
     + df['avoid_collaboration']
     + df['difficulty_controlling_emotions']) / 20
) * 100


# 4. Physical and Safety Risk Index (PSRI)
# 2 questions × maximum score of 5 = 10
# Divide by 10 and multiply by 100 → maximum possible score = 100
df['Physical_and_safety_risk_index(PSRI)'] = (
    (df['physically_tired_at_work']
     + df['accident_due_to_tiredness']) / 10
) * 100


# 5. Presentative Index (PI)
output_loss_map = {
    'About the same or better': 0.00,
    '20–40% less': 0.30,
    '40–60% less': 0.50,
    '60–80% less': 0.70,
    '80–100% less':1.00
}

df['productivity_loss_fraction'] = (
    df['current_work_output'].map(output_loss_map)
)

df['Presentative_index(PI)'] = (
    (df['unproductive_workdays_due_to_sleep'] / 23)
    * df['productivity_loss_fraction']
    * 100
)

df['Presentative_index(PI)'] = (
    df['Presentative_index(PI)'].round(2)
)


# 6. Overall Sleep Productivity Impact Index (SPI)
# Each component has a maximum of 100.
# Therefore, their total maximum = 400.
# Dividing by 4 gives an average with a maximum of 100.
df['Overall_sleep_Productivity_Impact_index(SPI)'] = (
    df['Cognitive_at_task_perforamnce_index_(CTPI)']
    + df['Emotional_and_social_impact_index(ESII)']
    + df['Physical_and_safety_risk_index(PSRI)']
    + df['Presentative_index(PI)']
) / 4

In [16]:
# data validation

df.describe()

,wake_up_refreshed,difficulty_falling_asleep,age,wake_up_multiple_times,weekend_sleep_schedule_difference,device_use_before_sleep,difficulty_concentrating,careless_mistakes,reread_due_to_focus_loss,difficulty_learning_new_skills,...,accident_due_to_tiredness,unproductive_workdays_due_to_sleep,wake_up_refreshed_reverse,sleep_quality_and_hygiene_index_(SQLI),Cognitive_at_task_perforamnce_index_(CTPI),Emotional_and_social_impact_index(ESII),Physical_and_safety_risk_index(PSRI),productivity_loss_fraction,Presentative_index(PI),Overall_sleep_Productivity_Impact_index(SPI)
count,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,...,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000
mean,2.921260,3.700787,24.303150,2.661417,3.700787,4.062992,2.677165,3.259843,3.377953,2.692913,...,2.464567,8.637795,3.078740,68.818898,56.587874,57.480315,56.456693,0.486614,18.521024,47.261476
std,0.956235,1.129088,10.050499,1.292349,1.236450,1.219862,1.174341,1.169809,1.119305,1.224821,...,1.132460,5.588635,0.956235,12.358030,15.971627,16.569646,18.193299,0.318320,18.720314,11.629570
min,1.000000,1.000000,10.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,0.000000,1.000000,36.000000,20.000000,20.000000,20.000000,0.000000,0.000000,21.667500
25%,2.000000,3.000000,19.000000,2.000000,3.000000,3.000000,2.000000,2.000000,3.000000,2.000000,...,2.000000,4.500000,2.000000,60.000000,46.670000,45.000000,45.000000,0.300000,3.910000,38.731250
50%,3.000000,4.000000,20.000000,3.000000,4.000000,5.000000,3.000000,3.000000,3.000000,2.000000,...,2.000000,7.000000,3.000000,72.000000,56.670000,55.000000,60.000000,0.500000,13.040000,47.157500
75%,4.000000,5.000000,21.500000,4.000000,5.000000,5.000000,3.000000,4.000000,4.000000,3.500000,...,3.000000,13.500000,4.000000,78.000000,66.670000,70.000000,70.000000,0.700000,30.430000,54.583750
max,5.000000,5.000000,54.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,...,5.000000,23.000000,5.000000,96.000000,100.000000,100.000000,100.000000,1.000000,100.000000,76.722500
